# Spectrogram-Based Audio Comparison

This notebook compares two audio files using **spectrograms** instead of chromagrams.

**Purpose:** Serve as a counter-example to the chromagram-based approach used in our main module.

### Key Differences:
- **Spectrogram**: Shows full frequency content over time (linear/log frequency scale)
- **Chromagram**: Folds all octaves into 12 pitch classes (C, C#, D, ... B)

Spectrograms are more detailed but less pitch-invariant, making them potentially less suitable for humming-to-song matching.

In [ ]:
# Import libraries and backend modules
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import os

# Import our backend modules
from match import preprocess_chroma, get_dtw_distance, SAMPLE_RATE

In [ ]:
# Configuration
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128  # Number of mel bands for mel-spectrogram

# Set your audio file paths here - Testing الشبيه vs Wala Ala Balo
AUDIO_FILE_1 = "الشبيه.mp3"
AUDIO_FILE_2 = r"songs\Amr Diab - Wala Ala Balo _ Official Music Video _ عمرو دياب - ولا على باله [PtY5863gINw].mp3"

In [ ]:
def load_audio(file_path, duration=30):
    """
    Load and preprocess an audio file (same as match.py approach).
    """
    print(f"Loading: {file_path}")
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True, duration=duration)
    
    # Trim silence (same as match.py)
    y, _ = librosa.effects.trim(y, top_db=20)
    
    # Normalize (same as match.py)
    y = librosa.util.normalize(y)
    
    print(f"  Duration: {len(y)/sr:.2f}s, Samples: {len(y)}")
    return y, sr

In [ ]:
def compute_spectrogram(y, sr):
    """
    Compute the Short-Time Fourier Transform (STFT) spectrogram.
    Returns magnitude spectrogram in dB scale.
    """
    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    spectrogram = np.abs(stft)
    spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)
    return spectrogram_db


def compute_mel_spectrogram(y, sr):
    """
    Compute Mel-frequency spectrogram.
    Better suited for perceptual audio analysis.
    """
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    return mel_spec_db


def compute_chromagram(y, sr):
    """
    Compute chromagram - same as our main module (match.py).
    """
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    return chroma

## Load Audio Files

In [ ]:
# Load both audio files
# Replace the file paths with your actual audio files

# Example: Use files from the songs folder or provide full paths
audio1_path = AUDIO_FILE_1
audio2_path = AUDIO_FILE_2

# Check if files exist
if not os.path.exists(audio1_path):
    print(f"⚠️ File not found: {audio1_path}")
    print("Please update AUDIO_FILE_1 with a valid path")
else:
    y1, sr1 = load_audio(audio1_path)

if not os.path.exists(audio2_path):
    print(f"⚠️ File not found: {audio2_path}")
    print("Please update AUDIO_FILE_2 with a valid path")
else:
    y2, sr2 = load_audio(audio2_path)

## Compute & Visualize Spectrograms

In [ ]:
# Compute all representations for both files
# (Only run if files were loaded successfully)

if 'y1' in dir() and 'y2' in dir():
    # STFT Spectrograms
    spec1 = compute_spectrogram(y1, sr1)
    spec2 = compute_spectrogram(y2, sr2)
    
    # Mel Spectrograms
    mel1 = compute_mel_spectrogram(y1, sr1)
    mel2 = compute_mel_spectrogram(y2, sr2)
    
    # Chromagrams (for comparison)
    chroma1 = compute_chromagram(y1, sr1)
    chroma2 = compute_chromagram(y2, sr2)
    
    print("✅ All features computed successfully!")
    print(f"\nSpectrograms shape: {spec1.shape}, {spec2.shape}")
    print(f"Mel spectrograms shape: {mel1.shape}, {mel2.shape}")
    print(f"Chromagrams shape: {chroma1.shape}, {chroma2.shape}")
else:
    print("❌ Please load audio files first!")

In [ ]:
def plot_comparison(feature1, feature2, title1, title2, feature_type='spectrogram'):
    """
    Plot two features side by side for comparison.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    if feature_type == 'spectrogram':
        img1 = librosa.display.specshow(
            feature1, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='log', ax=axes[0]
        )
        img2 = librosa.display.specshow(
            feature2, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='log', ax=axes[1]
        )
    elif feature_type == 'mel':
        img1 = librosa.display.specshow(
            feature1, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='mel', ax=axes[0]
        )
        img2 = librosa.display.specshow(
            feature2, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='mel', ax=axes[1]
        )
    elif feature_type == 'chroma':
        img1 = librosa.display.specshow(
            feature1, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='chroma', ax=axes[0]
        )
        img2 = librosa.display.specshow(
            feature2, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis='time', y_axis='chroma', ax=axes[1]
        )
    
    axes[0].set_title(title1)
    axes[1].set_title(title2)
    
    fig.colorbar(img1, ax=axes[0], format='%+2.0f dB')
    fig.colorbar(img2, ax=axes[1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize STFT Spectrograms
if 'spec1' in dir() and 'spec2' in dir():
    plot_comparison(spec1, spec2, 'Audio 1 - STFT Spectrogram', 'Audio 2 - STFT Spectrogram', 'spectrogram')

In [ ]:
# Visualize Mel Spectrograms
if 'mel1' in dir() and 'mel2' in dir():
    plot_comparison(mel1, mel2, 'Audio 1 - Mel Spectrogram', 'Audio 2 - Mel Spectrogram', 'mel')

In [ ]:
# Visualize Chromagrams (our main module's approach)
if 'chroma1' in dir() and 'chroma2' in dir():
    plot_comparison(chroma1, chroma2, 'Audio 1 - Chromagram', 'Audio 2 - Chromagram', 'chroma')

## DTW-Based Similarity Comparison

We'll use Dynamic Time Warping (DTW) to compare the audio files using different feature representations.

In [ ]:
def compute_dtw_distance_simple(feature1, feature2):
    """
    Wrapper around our backend's DTW - returns more info for visualization.
    """
    D, wp = librosa.sequence.dtw(feature1, feature2, metric='cosine', subseq=True)
    
    # Get the best alignment cost (same logic as match.py)
    best_row, best_col = wp[0]
    min_cost = D[best_row, best_col]
    path_length = len(wp)
    
    normalized_dist = min_cost / path_length
    return normalized_dist, wp, D

In [ ]:
# Compare using different feature types
if 'spec1' in dir() and 'spec2' in dir():
    print("=" * 60)
    print("DTW Distance Comparison (Lower = More Similar)")
    print("=" * 60)
    
    # Preprocess using our backend's preprocess_chroma function
    spec1_proc = preprocess_chroma(spec1)
    spec2_proc = preprocess_chroma(spec2)
    mel1_proc = preprocess_chroma(mel1)
    mel2_proc = preprocess_chroma(mel2)
    chroma1_proc = preprocess_chroma(chroma1)
    chroma2_proc = preprocess_chroma(chroma2)
    
    # Compute DTW distances
    dist_spec, _, _ = compute_dtw_distance_simple(spec1_proc, spec2_proc)
    dist_mel, _, _ = compute_dtw_distance_simple(mel1_proc, mel2_proc)
    
    # For chromagram, use our backend's get_dtw_distance (with penalty logic)
    dist_chroma_backend = get_dtw_distance(chroma1_proc, chroma2_proc)
    dist_chroma_simple, _, _ = compute_dtw_distance_simple(chroma1_proc, chroma2_proc)
    
    print(f"\n📊 STFT Spectrogram DTW Distance:  {dist_spec:.6f}")
    print(f"📊 Mel Spectrogram DTW Distance:   {dist_mel:.6f}")
    print(f"📊 Chromagram DTW Distance:        {dist_chroma_simple:.6f}")
    print(f"📊 Chromagram (with penalty):      {dist_chroma_backend:.6f}  ← match.py approach")
    
    print("\n" + "=" * 60)
    print("Analysis:")
    print("=" * 60)
    print("\n• Spectrograms capture detailed frequency information")
    print("  but are sensitive to pitch differences (different keys)")
    print("\n• Chromagrams fold octaves together, making them")
    print("  more robust to octave shifts (better for humming)")
    print("\n• The backend's get_dtw_distance adds stretch penalties")
    print("  to avoid matching short hums to long song segments")
else:
    print("Please load audio files first!")

## Visualize DTW Alignment Path

In [ ]:
def plot_dtw_alignment(D, wp, title):
    """
    Visualize the DTW cost matrix and optimal alignment path.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    img = ax.imshow(D, origin='lower', cmap='viridis', aspect='auto')
    
    # Plot the alignment path
    wp_array = np.array(wp)
    ax.plot(wp_array[:, 1], wp_array[:, 0], 'r-', linewidth=2, label='Warping Path')
    
    ax.set_xlabel('Audio 2 (frames)')
    ax.set_ylabel('Audio 1 (frames)')
    ax.set_title(title)
    ax.legend()
    
    fig.colorbar(img, ax=ax, label='Accumulated Cost')
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize DTW alignment for mel spectrogram
if 'mel1_proc' in dir() and 'mel2_proc' in dir():
    _, wp_mel, D_mel = compute_dtw_distance_simple(mel1_proc, mel2_proc)
    plot_dtw_alignment(D_mel, wp_mel, 'DTW Alignment - Mel Spectrogram')

In [ ]:
# Visualize DTW alignment for chromagram
if 'chroma1_proc' in dir() and 'chroma2_proc' in dir():
    _, wp_chroma, D_chroma = compute_dtw_distance_simple(chroma1_proc, chroma2_proc)
    plot_dtw_alignment(D_chroma, wp_chroma, 'DTW Alignment - Chromagram')

## Summary & Observations

### Spectrogram vs Chromagram for Humming Detection:

| Feature | Pros | Cons |
|---------|------|------|
| **STFT Spectrogram** | Full frequency detail | Sensitive to pitch/key changes |
| **Mel Spectrogram** | Perceptually weighted | Still sensitive to octave shifts |
| **Chromagram** | Pitch-class representation | Loses octave information |

### Why Chromagrams Work Better for Humming:

1. **Key Invariance**: Humming is often in a different key than the original song
2. **Octave Folding**: People hum in comfortable ranges, often different octaves
3. **Melody Focus**: Chromagrams emphasize harmonic/melodic content over timbral details

### When Spectrograms Might Be Better:

1. **Same recording comparison**: Exact or near-exact audio matching
2. **Timbre-based matching**: When sound texture matters
3. **Audio fingerprinting**: Where exact frequency patterns are important